# Étude d'un circuit RC avec Arduino

Ce notebook permet de suivre l'évolution de la tension aux bornes d'un condensateur après une modification de la tension délivrée par une voie numérique. Il utilise directement le code `acquisition_continue.ino` téléversé sur la carte Arduino.

In [ ]:
# Import des bibliothèques nécessaires

%run Import_bibliotheques.ipynb

%run Arduino.ipynb

## Conditions expérimentales :

In [ ]:
charge = 1    # mettre 1 pour la charge du condensateur, et 0 pour sa décharge
duree = 0.2    # Entrer la durée d'acquisition désirée en s
Analog_pin = 0    # numéro de la voie de mesure analogique (déjà configurée dans le code Arduino)
Digital_pin = 2    # numéro de la voie de commande digitale (doit être entre 2 et 5 pour notre code Arduino)

## Connexion à Arduino et acquisition des données

In [ ]:
# Initialisation des listes pour stocker les données
uc = []
t = []

# Détection et connexion à l'Arduino
try:
    connexion_arduino()
    print("Connexion établie avec Arduino")
except Exception as e:
    print(f"Erreur de connexion: {e}")
    raise

In [ ]:
# Fonction pour basculer une broche digitale
def set_digital_pin(pin, state):
    command = f"D{pin}:{state}\n"
    ser.write(command.encode())
    # L'Arduino émet en continu : on lit les lignes jusqu'à la réponse à la commande,
    # mais on n'y passe JAMAIS plus d'une seconde (commande envoyée même sans réponse)
    debut = time()
    while time() - debut < 1.0 :
        ligne = ser.readline()
        if not ligne :
            continue
        reponse = ligne.decode('utf-8', errors='replace').strip()
        if reponse.startswith(("OK:", "ERREUR:")) :
            print(f"Réponse Arduino: {reponse}")
            return
    print("Pas de réponse confirmée de l'Arduino (commande envoyée quand même).")

In [ ]:
# Reconnexion automatique si le port est fermé
if 'ser' not in globals() or not ser.is_open :
    connexion_arduino()

print(f"Condition initiale: {'charge' if charge == 1 else 'décharge'}")

# Mise en place de la condition initiale (inverse de ce qu'on va faire ensuite)
set_digital_pin(Digital_pin, 1 - charge)
sleep(0.5)  # Attente pour stabilisation

# Vidage du buffer série
ser.reset_input_buffer()

# Basculement de la broche digitale pour démarrer la charge/décharge
print(f"Début de l'acquisition pour {duree} secondes...")
set_digital_pin(Digital_pin, charge)

# Acquisition : l'horodatage Arduino (µs) sert de référence temporelle
t = []
uc = []
temps_initial = None
lectures_vides = 0
debut_acquisition = time()
while True:
    try:
        ligne = ser.readline()
        parse = parse_ligne(ligne)    # format "temps_us,tension" ; None si ligne invalide
        if parse is None:
            if ligne:
                lectures_vides = 0    # ligne reçue mais invalide : on continue
            else:
                lectures_vides += 1    # aucune donnée reçue
                if lectures_vides > 10 :
                    print("Plus de données reçues de l'Arduino : arrêt.")
                    break
            continue
        temps_us, tension = parse
        if temps_initial is None :
            temps_initial = temps_us
        t.append((temps_us - temps_initial) / 1e6)    # conversion en secondes
        uc.append(tension)
        if t[-1] >= duree :
            break
        # Filet de sécurité : on ne reste jamais bloqué au-delà de duree + 3 s
        if time() - debut_acquisition > duree + 3 :
            print("Durée maximale dépassée : arrêt.")
            break
    except KeyboardInterrupt:
        print("Acquisition interrompue.")
        break

if len(t) == 0 :
    raise Exception("Aucune donnée reçue : vérifier le câble USB et le programme acquisition_continue.ino.")

# Conversion des listes en tableaux numpy
t = array(t)
uc = array(uc)
print(f"Acquisition terminée : {len(t)} points collectés sur {t[-1]:.3f} s.")

# Sauvegarde des données (importables avec tableurVersVariables('rc.csv'))
with open("rc.csv", 'w') as fichier :
    fichier.write("t,uc\n")
    for temps_relatif, tension in zip(t, uc) :
        fichier.write(f"{temps_relatif:.6f},{tension:.6f}\n")
print("Données sauvegardées dans rc.csv.")

## Calcul des caractéristiques de la courbe de régression

In [ ]:
# Définition des contraintes pour le modèle
if charge == 0:
    equation = "E * exp(-1 * t / tau)"
else:
    equation = "E * (1 - exp(-1 * t / tau))"

# Contraintes sous forme de liste de tuples (nom_paramètre, valeur_initiale, vary, min, max)
contraintes = [
    ('tau', 1, True, 0, inf),
    ('E', 5, True, 0, 5.5)
]

# Calcul du modèle
xMod, yMod, expression, valeurs, abscisse, ordonnee, modele, parametres = Calcul_modele(
    't',           # abscisse
    'uc',          # ordonnée
    equation,      # équation
    0,             # début de la plage de données à utiliser
    len(t),        # fin de la plage de données à utiliser
    None,          # début de la courbe du modèle (None = min des données)
    None,          # fin de la courbe du modèle (None = max des données)
    contraintes    # contraintes sur les paramètres
)

## Tracé du graphique

In [ ]:
fig = go.Figure()

# Tracé des points expérimentaux :

fig.add_trace(go.Scatter(x=abscisse, y=ordonnee, mode="markers", name="Points expérimentaux",
    marker=dict(symbol="cross", size=10, color="blue")))

# Tracé du modèle :

fig.add_trace(go.Scatter(x=xMod, y=yMod, mode="lines", name=f"Modèle : {expression}<br>{valeurs.strip().replace(chr(10), '<br>')}",
    line=dict(color="crimson", width=2)))

# Cosmétique du graphique :
#fig.update_yaxes(scaleanchor="x", scaleratio=1)    # à décommenter pour un repère orthonormé
fig.update_xaxes(showgrid=True)    # grille verticale (par défaut)
fig.update_yaxes(showgrid=True)    # grille horizontale (par défaut)
fig.update_layout(
    showlegend=True,    # légende du graphique,
    title="Variation de la tension aux bornes du condensateur",    # mettre le titre désiré entre les guillemets,
    xaxis_title="t (s)",    # nom de l'axe des abscisses,
    yaxis_title="uc (V)",    # nom de l'axe des ordonnées,
    height=600)    # hauteur du graphique en pixels ; ex. : 450, 700...


## Calcul de la valeur de la capacité du condensateur

In [ ]:
R = 220    # Entrer la valeur de la résistance en Ohm
c = tau / R
print(f"La valeur de la capacité du condensateur est {c:.3g} F")
incert_tau = parametres['tau'].stderr    # incertitude-type du paramètre tau donnée par l'ajustement
if incert_tau is not None :
    incert_c = incert_tau / R    # propagation : C = tau / R
    print(f"avec une incertitude-type de {incert_c:.3g} F")

In [ ]:
# Fermeture de la connexion série
ser.close()
print("Connexion avec Arduino fermée.")